In [1]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from sklearn.metrics import get_scorer

dataset=AlexMI()

paradigm = MotorImagery(n_classes=len(dataset.event_id),resample=250)    

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)
scorer = get_scorer(paradigm.scoring)


Choosing from all possible events


In [2]:
from sklearn.model_selection import StratifiedKFold

n_blocks=16
cv = StratifiedKFold(n_splits=16, shuffle=True, random_state=42)
n_blocks_grid = list(range(1,n_blocks+1))
theta_grid = [0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1]
#theta_grid=[0,0.2,1]

In [3]:
job_args = []
for subject in dataset.subject_list[:1]:
    X, y, meta = paradigm.get_data(dataset=dataset, subjects=[subject], cache_config=cache_config,)
    for fold, (train_idc, test_idc) in enumerate(cv.split(X, y)):
        for theta in theta_grid:
            job_args.append((dataset.code, subject, fold, X,y, train_idc, test_idc, theta))

In [4]:
from hoda.hoda import BTTDA
from hoda.classification import ZScore, ZLogRatio
from hoda.tensorize import fh_power, fh_log_envelope
from sklearn.preprocessing import StandardScaler
from classification_mi import make_clf, get_hoda_params, get_bttda_params, stf_transform
import tensorly as tl
import pandas as pd
import warnings

def eval_fold(dataset, subject, fold, X, y, train_idc, test_idc, theta):
    X = stf_transform(X)
    X_st = ZScore().fit(X[train_idc], y[train_idc]).transform(X,y) 
    
    hoda_params = get_hoda_params()
    hoda_params['theta'] = theta
    bttda = BTTDA(
        ranks=[None]*max(n_blocks_grid),
        hoda_params=hoda_params,
        verbose=False,        
    )
    
    bttda.fit(X_st[train_idc], y[train_idc])
    print(bttda.n_blocks_)
    clf = make_clf()
    result = []
    for n_blocks in n_blocks_grid:
        if n_blocks > bttda.n_blocks_:
            break
        Xt = bttda.transform(X_st, n_blocks=n_blocks)
        X_rec = bttda.inv_transform(Xt, n_blocks=n_blocks)
        try:
            clf.fit(Xt[train_idc], y[train_idc])
        except ValueError as e:
            warnings.warn(str(e))
            break
        result.append(dict(
            subject = subject,
            dataset = dataset,
            fold = fold,
            theta=theta,
            n_blocks=n_blocks,
            train_score = scorer(clf, Xt[train_idc], y[train_idc]),
            test_score = scorer(clf, Xt[test_idc], y[test_idc]),
            train_mse = tl.metrics.regression.MSE(X_st[train_idc], X_rec[train_idc]),
            test_mse = tl.metrics.regression.MSE(X_st[test_idc], X_rec[test_idc]),          
        ))
    return pd.DataFrame(result)

        

In [5]:
import joblib
from joblib import Parallel, delayed
from hpc import create_cluster, create_client, TIMEOUT

with create_cluster(cluster='cpu') as cluster, create_client(cluster) as client:
    with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
        results = Parallel(n_jobs=len(job_args), verbose=True)(delayed(eval_fold)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

[Parallel(n_jobs=176)]: Using backend DaskDistributedBackend with 9 concurrent workers.
[Parallel(n_jobs=176)]: Done  32 tasks      | elapsed:  4.0min
[Parallel(n_jobs=176)]: Done 176 out of 176 | elapsed: 16.7min finished


In [6]:
import plotly.io as pio
pio.renderers.default = 'iframe'

In [7]:
results.to_csv('results/gridsearch_mi.csv')

In [8]:
results

,subject,dataset,fold,theta,n_blocks,train_score,test_score,train_mse,test_mse
0,1,AlexandreMotorImagery,0,0.0,1,0.857143,0.00,0.994717,1.044503
1,1,AlexandreMotorImagery,0,0.0,2,0.964286,0.25,0.992634,1.043012
2,1,AlexandreMotorImagery,0,0.0,3,1.000000,0.50,0.990039,1.041602
3,1,AlexandreMotorImagery,0,0.0,4,1.000000,0.50,0.988355,1.041071
4,1,AlexandreMotorImagery,0,0.0,5,1.000000,0.75,0.986766,1.040253
...,...,...,...,...,...,...,...,...,...
2651,1,AlexandreMotorImagery,15,1.0,2,1.000000,0.00,0.000000,0.000000
2652,1,AlexandreMotorImagery,15,1.0,3,1.000000,0.00,0.000000,0.000000
2653,1,AlexandreMotorImagery,15,1.0,4,1.000000,0.00,0.000000,0.000000
2654,1,AlexandreMotorImagery,15,1.0,5,1.000000,0.00,0.000000,0.000000


In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

df = results.groupby(['n_blocks', 'theta'])['test_score'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='test_score', color='theta', color_discrete_sequence=colors)
fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = results.groupby(['n_blocks', 'theta'])['test_mse'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='test_mse', color='theta', color_discrete_sequence=colors)
fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = results.groupby(['n_blocks', 'theta'])['train_mse'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='train_mse', color='theta', color_discrete_sequence=colors)
fig.show()